In [ ]:

import json, random
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sentence_transformers import SentenceTransformer

# sklearn
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import KBinsDiscretizer

In [ ]:
from huggingface_hub import login

# This will prompt you for your token securely
login()


In [ ]:
# ---------------------------------------------------------------
# 1. LOAD DATA
# ---------------------------------------------------------------

train_df = pd.read_json("train_data.json")
test_df = pd.read_json("test_data.json")

def build_text(row):
    sys = row.get("system_prompt", "")
    return f"{sys} {row['user_prompt']} {row['response']}"

train_texts = train_df.apply(build_text, axis=1).tolist()
test_texts  = test_df.apply(build_text, axis=1).tolist()

In [ ]:
# ---------------------------------------------------------------
# 2. BUILD TEXT EMBEDDINGS (GEMMA 300M)
# ---------------------------------------------------------------

print("Loading Gemma embedding model...")
embedder = SentenceTransformer("google/embeddinggemma-300m")

print("Encoding training texts with Gemma...")
train_text_embeddings = embedder.encode(
    train_texts, convert_to_numpy=True, show_progress_bar=True
)

print("Encoding test texts with Gemma...")
test_text_embeddings = embedder.encode(
    test_texts, convert_to_numpy=True, show_progress_bar=True
)

np.save("train_text_embeddings.npy", train_text_embeddings)
np.save("test_text_embeddings.npy", test_text_embeddings)

# ---------------------------------------------------------------
# 3. METRIC NAME EMBEDDINGS (GEMMA 300M)
# ---------------------------------------------------------------

if "metric_name" not in train_df.columns:
    raise ValueError("train_df must include a 'metric_name' column!")

metric_names = sorted(train_df["metric_name"].unique())

with open("metric_names.json", "w") as f:
    json.dump(metric_names, f, indent=2)

print("Encoding metric names with Gemma...")
metric_name_embeddings = embedder.encode(
    metric_names, convert_to_numpy=True, show_progress_bar=True
)

np.save("metric_name_embeddings.npy", metric_name_embeddings)

metric_embs = {m: metric_name_embeddings[i] for i, m in enumerate(metric_names)}

print("Embedding shapes:")
print("  train text:", train_text_embeddings.shape)
print("  test text: ", test_text_embeddings.shape)
print("  metric:    ", metric_name_embeddings.shape)


Loading Gemma embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Encoding training texts with Gemma...


Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Encoding test texts with Gemma...


Batches:   0%|          | 0/114 [00:00<?, ?it/s]

Encoding metric names with Gemma...


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Embedding shapes:
  train text: (5000, 768)
  test text:  (3638, 768)
  metric:     (145, 768)


In [ ]:
# ---------------------------------------------------------------
# 4. MODEL DEFINITION (YOUR ORIGINAL MODEL)
# ---------------------------------------------------------------

class MetricModel(nn.Module):
    def __init__(self, text_dim, metric_dim, hidden_dim=512):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(text_dim + metric_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, text_emb, metric_emb):
        x = torch.cat([text_emb, metric_emb], dim=-1)
        return self.fc(x)


# ---------------------------------------------------------------
# 5. LOSS FUNCTIONS
# ---------------------------------------------------------------

def info_nce_loss(pos_scores, neg_scores, temperature=0.1):
    """
    pos_scores: (B, 1)
    neg_scores: (B, K)
    """
    logits = torch.cat([pos_scores, neg_scores], dim=1)  # (B, K+1)
    logits = logits / temperature

    labels = torch.zeros(logits.size(0), dtype=torch.long, device=logits.device)

    loss = nn.CrossEntropyLoss()(logits, labels)
    return loss

def ranking_loss(pos_scores, neg_scores, margin=1.0):
    # pos_scores: (B,1)
    # neg_scores: (B,K)

    # Want: pos > neg + margin
    diffs = margin - (pos_scores - neg_scores)  # (B,K)
    loss = torch.relu(diffs).mean()
    return loss

# ---------------------------------------------------------------
# 6. CONTRASTIVE TRAINING  (FIXED VERSION)
# ---------------------------------------------------------------

def train_contrastive(model, optimizer, text_embs, metric_embs, metric_names,
                      epochs=5, batch_size=16, neg_k=3, device="cpu"):

    model.to(device)
    n = len(text_embs)

    dataset = TensorDataset(text_embs, torch.arange(n))
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    for epoch in range(epochs):
        total_loss = 0
        model.train()

        for text_batch, idx_batch in loader:
            text_batch = text_batch.to(device)
            B = len(idx_batch)

            # ----------------------------
            # POSITIVE METRIC EMBEDDINGS
            # ----------------------------
            pos_metric_batch = torch.tensor(
                [metric_embs[metric_names[i]] for i in idx_batch],
                dtype=torch.float32, device=device
            )

            # ----------------------------
            # NEGATIVE METRIC EMBEDDINGS
            # (Compute properly for InfoNCE)
            # ----------------------------
            neg_metric_list = []
            for _ in range(neg_k):
                rand_idx = torch.randint(0, n, (B,))
                neg_metric_list.append(
                    torch.tensor(
                        [metric_embs[metric_names[i]] for i in rand_idx],
                        dtype=torch.float32, device=device
                    )
                )

            neg_metrics = torch.stack(neg_metric_list, dim=1)  # (B, K, Dm)

            # text: (B, Dt) → (B,K,Dt)
            text_repeat = text_batch.unsqueeze(1).repeat(1, neg_k, 1)

            # flatten for forward pass
            neg_scores_flat = model(
                text_repeat.reshape(-1, text_batch.size(1)),
                neg_metrics.reshape(-1, pos_metric_batch.size(1))
            )

            # reshape back → (B, K)
            neg_scores = neg_scores_flat.reshape(B, neg_k)

            # positive → (B,1)
            pos_scores = model(text_batch, pos_metric_batch)

            optimizer.zero_grad()
            loss = ranking_loss(pos_scores, neg_scores)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"[Contrastive] Epoch {epoch+1}: loss = {total_loss/len(loader):.4f}")


# ---------------------------------------------------------------
# 7. REGRESSION TRAINING (unchanged)
# ---------------------------------------------------------------

def train_regression(model, optimizer, text_embs, metric_embs, metric_names,
                     scores, epochs=3, batch_size=16, device="cpu"):

    model.to(device)
    n = len(text_embs)

    dataset = TensorDataset(
        text_embs,
        torch.tensor(scores, dtype=torch.float32),
        torch.arange(n)
    )
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    loss_fn = nn.MSELoss()

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for text_batch, score_batch, idx_batch in loader:
            text_batch = text_batch.to(device)
            score_batch = score_batch.to(device)

            metric_batch = torch.tensor(
                [metric_embs[metric_names[i]] for i in idx_batch],
                dtype=torch.float32, device=device
            )

            optimizer.zero_grad()
            preds = torch.sigmoid(model(text_batch, metric_batch)) * 10
            loss = loss_fn(preds.squeeze(), score_batch)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"[Regression] Epoch {epoch+1}: loss = {total_loss/len(loader):.4f}")


# ---------------------------------------------------------------
# 8. PREDICTION FUNCTION
# ---------------------------------------------------------------

def predict(model, test_embs, test_metric_names, metric_embs, device="cpu"):
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(len(test_embs)):
            t = test_embs[i].unsqueeze(0).to(device)
            m = torch.tensor(metric_embs[test_metric_names[i]],
                             dtype=torch.float32).unsqueeze(0).to(device)
            s = torch.sigmoid(model(t, m)).item() * 10
            preds.append(s)
    return preds


# ---------------------------------------------------------------
# 9. TRAINING LOOP (same logic, no change)
# ---------------------------------------------------------------

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

text_embs_train = torch.tensor(train_text_embeddings, dtype=torch.float32)
text_embs_test  = torch.tensor(test_text_embeddings, dtype=torch.float32)

text_dim = train_text_embeddings.shape[1]
metric_dim = metric_name_embeddings.shape[1]
scores = train_df["score"].astype(float).values

bins = KBinsDiscretizer(n_bins=10, encode='ordinal', strategy='quantile')\
        .fit_transform(scores.reshape(-1, 1)).squeeze()

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

best_models = []
fold_results = []

for fold, (tr, va) in enumerate(skf.split(text_embs_train, bins)):
    print(f"\n===== Fold {fold+1} =====")

    model = MetricModel(text_dim, metric_dim).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    train_embs = text_embs_train[tr]
    val_embs   = text_embs_train[va]

    train_metric_names = train_df.iloc[tr]["metric_name"].tolist()
    val_metric_names   = train_df.iloc[va]["metric_name"].tolist()

    # contrastive
    train_contrastive(
        model, optimizer,
        text_embs=train_embs,
        metric_embs=metric_embs,
        metric_names=train_metric_names,
        epochs=10, batch_size=16, neg_k=3,
        device=device
    )

    # validation
    preds_val = []
    with torch.no_grad():
        for i in range(len(val_embs)):
            t = val_embs[i].unsqueeze(0).to(device)
            m = torch.tensor(metric_embs[val_metric_names[i]],
                             dtype=torch.float32).unsqueeze(0).to(device)
            s = torch.sigmoid(model(t, m)).item() * 10
            preds_val.append(s)

    mse = np.mean((np.array(preds_val) - scores[va]) ** 2)
    print(f"Fold {fold+1} MSE = {mse:.4f}")

    fold_results.append(mse)
    best_models.append(model.state_dict())


best_fold = int(np.argmin(fold_results))
print("\nBEST FOLD:", best_fold+1, "MSE =", fold_results[best_fold])

model = MetricModel(text_dim, metric_dim)
model.load_state_dict(best_models[best_fold])
model.to(device)

test_metric_names = test_df["metric_name"].tolist()

preds = predict(
    model,
    text_embs_test,
    test_metric_names,
    metric_embs,
    device=device
)

sub = pd.read_csv("sample_submission.csv")
sub["score"] = preds
sub.to_csv("submission.csv", index=False)

print("\nSaved submission.csv ✓")


Using device: cuda

===== Fold 1 =====


/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_discretization.py:306: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 0 are removed. Consider decreasing the number of bins.
  warnings.warn(
/tmp/ipython-input-2298332302.py:69: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  pos_metric_batch = torch.tensor(


[Contrastive] Epoch 1: loss = 0.6581
[Contrastive] Epoch 2: loss = 0.2478
[Contrastive] Epoch 3: loss = 0.1705
[Contrastive] Epoch 4: loss = 0.1349
[Contrastive] Epoch 5: loss = 0.1086
[Contrastive] Epoch 6: loss = 0.0931
[Contrastive] Epoch 7: loss = 0.0819
[Contrastive] Epoch 8: loss = 0.0760
[Contrastive] Epoch 9: loss = 0.0673
[Contrastive] Epoch 10: loss = 0.0631
Fold 1 MSE = 8.3351

===== Fold 2 =====
[Contrastive] Epoch 1: loss = 0.6627
[Contrastive] Epoch 2: loss = 0.2618
[Contrastive] Epoch 3: loss = 0.1735
[Contrastive] Epoch 4: loss = 0.1355
[Contrastive] Epoch 5: loss = 0.1161
[Contrastive] Epoch 6: loss = 0.0965
[Contrastive] Epoch 7: loss = 0.0837
[Contrastive] Epoch 8: loss = 0.0767
[Contrastive] Epoch 9: loss = 0.0701
[Contrastive] Epoch 10: loss = 0.0602
Fold 2 MSE = 5.6618

===== Fold 3 =====
[Contrastive] Epoch 1: loss = 0.6520
[Contrastive] Epoch 2: loss = 0.2504
[Contrastive] Epoch 3: loss = 0.1735
[Contrastive] Epoch 4: loss = 0.1297
[Contrastive] Epoch 5: loss = 

In [ ]:
# ============================================================
# Dual-Head Alignment Model — Full Pipeline (NO CACHE VERSION)
# ============================================================

import os
import numpy as np
import pandas as pd
import json
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

# -----------------------------
# CONFIG
# -----------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH = 16
LOW_THRESHOLD = 0.5


# -----------------------------
# DATASET (NO CACHING)
# -----------------------------
class MetricDataset(Dataset):
    def __init__(self, df, metric_embs, embedder, has_score=True):
        self.df = df.reset_index(drop=True)
        self.metric_embs = metric_embs
        self.embedder = embedder
        self.has_score = has_score

        print("Encoding dataset... (NO CACHE)")
        embs = []

        for _, row in tqdm(self.df.iterrows(), total=len(self.df)):
            parts = [
                str(row.get("system_prompt", "")),
                str(row.get("user_prompt", "")),
                str(row.get("response", ""))
            ]

            vecs = self.embedder.encode(parts, convert_to_numpy=True)
            merged = np.concatenate(vecs, axis=0)
            embs.append(merged)

        arr = np.vstack(embs)
        self.text_embs = torch.tensor(arr, dtype=torch.float32)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        text_vec = self.text_embs[idx]
        metric_vec = torch.tensor(self.metric_embs[row["metric_name"]], dtype=torch.float32)

        score = float(row["score"]) / 10.0 if self.has_score else 0.0

        return text_vec, metric_vec, torch.tensor(score, dtype=torch.float32)


# -----------------------------
# MODEL
# -----------------------------
class DualHeadAlignmentModel(nn.Module):
    def __init__(self, text_dim, metric_dim, hidden=512):
        super().__init__()

        self.text_proj = nn.Sequential(
            nn.Linear(text_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden)
        )

        self.metric_proj = nn.Sequential(
            nn.Linear(metric_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden)
        )

        self.rank_head = nn.Linear(hidden, 1)

        self.reg_low = nn.Sequential(
            nn.Linear(text_dim + metric_dim, hidden // 2),
            nn.ReLU(),
            nn.Linear(hidden // 2, 1)
        )

        self.reg_high = nn.Sequential(
            nn.Linear(text_dim + metric_dim, hidden // 2),
            nn.ReLU(),
            nn.Linear(hidden // 2, 1)
        )

    def embed(self, t, m):
        tp = self.text_proj(t)
        mp = self.metric_proj(m)
        return tp, mp

    def forward_rank(self, t, m):
        tp, mp = self.embed(t, m)
        return self.rank_head(tp * mp).squeeze(-1)

    def forward_reg_low(self, t, m):
        return self.reg_low(torch.cat([t, m], dim=-1)).squeeze(-1)

    def forward_reg_high(self, t, m):
        return self.reg_high(torch.cat([t, m], dim=-1)).squeeze(-1)


# -----------------------------
# TRAINING
# -----------------------------
def train(model, loader, epochs=5, lr=2e-4, low_thresh=LOW_THRESHOLD):
    optim = torch.optim.Adam(model.parameters(), lr=lr)
    bce = nn.BCEWithLogitsLoss()
    mse = nn.MSELoss()

    for ep in range(epochs):
        model.train()
        total = 0.0

        for text, metric, score in loader:
            text, metric, score = text.to(DEVICE), metric.to(DEVICE), score.to(DEVICE)

            optim.zero_grad()

            tp, mp = model.embed(text, metric)
            sim = (tp * mp).sum(dim=-1)

            mask_high = score > low_thresh
            mask_low = score <= low_thresh

            # ---- Contrastive losses ----
            if mask_high.any():
                loss_pos = bce(sim[mask_high], torch.ones_like(sim[mask_high]))
            else:
                loss_pos = 0.0 * sim.sum()

            perm = torch.randperm(metric.size(0))
            sim_neg_swap = (tp * model.metric_proj(metric[perm])).sum(dim=-1)
            loss_neg_swapped = bce(sim_neg_swap, torch.zeros_like(sim_neg_swap))

            if mask_low.any():
                loss_neg_low = bce(sim[mask_low], torch.zeros_like(sim[mask_low]))
            else:
                loss_neg_low = 0.0 * sim.sum()

            # ---- Regression losses ----
            loss_reg = 0.0
            if mask_low.any():
                reg_low = torch.sigmoid(model.forward_reg_low(text[mask_low], metric[mask_low]))
                loss_reg += mse(reg_low, score[mask_low])
            if mask_high.any():
                reg_high = torch.sigmoid(model.forward_reg_high(text[mask_high], metric[mask_high]))
                loss_reg += mse(reg_high, score[mask_high])

            loss = loss_pos + loss_neg_swapped + loss_neg_low + loss_reg
            loss.backward()
            optim.step()

            total += loss.item()

        print(f"Epoch {ep+1}: loss = {total/len(loader):.4f}")


# -----------------------------
# PREDICTION
# -----------------------------
def predict(model, loader, thresh=LOW_THRESHOLD):
    model.eval()
    out = []

    with torch.no_grad():
        for text, metric, _ in loader:
            text, metric = text.to(DEVICE), metric.to(DEVICE)

            tp, mp = model.embed(text, metric)
            sim = torch.sigmoid((tp * mp).sum(dim=-1))

            mask_low = sim <= thresh
            mask_high = sim > thresh

            if mask_low.any():
                raw = model.forward_reg_low(text[mask_low], metric[mask_low])
                sim[mask_low] = torch.sigmoid(raw)

            if mask_high.any():
                raw = model.forward_reg_high(text[mask_high], metric[mask_high])
                sim[mask_high] = torch.sigmoid(raw)

            out.extend((sim.cpu().numpy() * 10).tolist())

    return out


# ============================================================
# MAIN EXECUTION BLOCK
# ============================================================

print("Device:", DEVICE)

train_df = pd.read_json("train_data.json")
test_df = pd.read_json("test_data.json")

metric_names = json.load(open("metric_names.json"))
metric_embs_arr = np.load("metric_name_embeddings.npy")

metric_embs = {m: metric_embs_arr[i] for i, m in enumerate(metric_names)}

embedder = SentenceTransformer("sentence-transformers/multi-qa-mpnet-base-dot-v1")

train_ds = MetricDataset(train_df, metric_embs, embedder, has_score=True)
test_ds = MetricDataset(test_df, metric_embs, embedder, has_score=False)

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=BATCH)

text_dim = train_ds.text_embs.shape[1]
metric_dim = len(next(iter(metric_embs.values())))


Device: cuda
Encoding dataset... (NO CACHE)


100%|██████████| 5000/5000 [05:16<00:00, 15.81it/s]


Encoding dataset... (NO CACHE)


100%|██████████| 3638/3638 [03:48<00:00, 15.92it/s]


In [ ]:

model = DualHeadAlignmentModel(text_dim, metric_dim).to(DEVICE)

EPOCHS = 5
train(model, train_loader, epochs=EPOCHS)

preds = predict(model, test_loader)
preds_train = predict(model, train_loader)

os.makedirs("submissions", exist_ok=True)

sub = pd.read_csv("sample_submission.csv")
sub["score"] = preds
path = f"submissions/submission_dualhead_nocache_{EPOCHS}ep.csv"
sub.to_csv(path, index=False)
print("Saved:", path)

train_df["pred_score"] = preds_train
train_df.to_csv("train_df_with_preds_nocache.csv", index=False)
print("Saved train_df_with_preds_nocache.csv")


Epoch 1: loss = 1.2701
Epoch 2: loss = 0.9874
Epoch 3: loss = 0.8556
Epoch 4: loss = 0.7822
Epoch 5: loss = 0.6966
Saved: submissions/submission_dualhead_nocache_5ep.csv
Saved train_df_with_preds_nocache.csv


In [ ]:
# ============================================================
# 🔥 UPGRADED TRAINING + PREDICTION PIPELINE (NO LOSS CHANGE)
# ============================================================

import torch.nn.functional as F

# -----------------------------
# Improved Training Loop
# -----------------------------
def train_upgraded(
    model,
    loader,
    epochs=6,
    lr=2e-4,
    weight_decay=0.01,
    grad_accum=1,
    clip=1.0,
    low_thresh=LOW_THRESHOLD,
):
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    total_steps = epochs * len(loader)
    warmup_steps = int(0.1 * total_steps)
    step = 0

    for ep in range(epochs):
        total_loss = 0.0

        for i, (text, metric, score) in enumerate(loader):
            text, metric, score = text.to(DEVICE), metric.to(DEVICE), score.to(DEVICE)
            optimizer.zero_grad()

            tp, mp = model.embed(text, metric)
            sim = (tp * mp).sum(dim=-1)

            mask_high = score > low_thresh
            mask_low  = score <= low_thresh

            # ---- POSITIVE CONTRASTIVE ----
            if mask_high.any():
                loss_pos = F.binary_cross_entropy_with_logits(
                    sim[mask_high], torch.ones_like(sim[mask_high])
                )
            else:
                loss_pos = 0.0 * sim.sum()

            # ---- NEG SWAP ----
            perm = torch.randperm(metric.size(0))
            sim_neg_swap = (tp * model.metric_proj(metric[perm])).sum(dim=-1)
            loss_neg_swap = F.binary_cross_entropy_with_logits(
                sim_neg_swap, torch.zeros_like(sim_neg_swap)
            )

            # ---- NEG LOW ----
            if mask_low.any():
                loss_neg_low = F.binary_cross_entropy_with_logits(
                    sim[mask_low], torch.zeros_like(sim[mask_low])
                )
            else:
                loss_neg_low = 0.0 * sim.sum()

            # --------------------
            # REGRESSION HEADS
            # --------------------
            loss_reg = 0.0

            if mask_low.any():
                pred_low = torch.sigmoid(model.forward_reg_low(text[mask_low], metric[mask_low]))
                loss_reg += F.mse_loss(pred_low, score[mask_low])

            if mask_high.any():
                pred_high = torch.sigmoid(model.forward_reg_high(text[mask_high], metric[mask_high]))
                loss_reg += F.mse_loss(pred_high, score[mask_high])

            loss = loss_pos + loss_neg_swap + loss_neg_low + loss_reg
            loss.backward()

            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip)

            optimizer.step()

            # Warmup schedule
            step += 1
            if step < warmup_steps:
                lr_scale = step / warmup_steps
                for pg in optimizer.param_groups:
                    pg["lr"] = lr * lr_scale

            total_loss += loss.item()

        print(f"Epoch {ep+1}/{epochs} — loss: {total_loss/len(loader):.4f}")


# -----------------------------
# Improved Prediction
# -----------------------------
def predict_upgraded(model, loader, low_thresh=LOW_THRESHOLD):
    model.eval()
    outputs = []

    with torch.no_grad():
        for text, metric, _ in loader:
            text, metric = text.to(DEVICE), metric.to(DEVICE)

            # Ranking via sim
            tp, mp = model.embed(text, metric)
            sim = torch.sigmoid((tp * mp).sum(dim=-1))

            mask_low  = sim <= low_thresh
            mask_high = sim > low_thresh

            if mask_low.any():
                raw_low = model.forward_reg_low(text[mask_low], metric[mask_low])
                sim[mask_low] = torch.sigmoid(raw_low)

            if mask_high.any():
                raw_high = model.forward_reg_high(text[mask_high], metric[mask_high])
                sim[mask_high] = torch.sigmoid(raw_high)

            outputs.extend((sim.cpu().numpy() * 10).tolist())

    return outputs


# ============================================================
# RUN TRAINING
# ============================================================

EPOCHS = 10
train_upgraded(model, train_loader, epochs=EPOCHS)

# ============================================================
# RUN PREDICTION
# ============================================================

preds_test = predict_upgraded(model, test_loader)
preds_train = predict_upgraded(model, train_loader)

os.makedirs("submissions", exist_ok=True)

sub = pd.read_csv("sample_submission.csv")
sub["score"] = preds_test
out_path = f"submissions/submission_dualhead_upgraded_{EPOCHS}ep.csv"
sub.to_csv(out_path, index=False)
print("Saved:", out_path)

train_df["pred_score"] = preds_train
train_df.to_csv("train_df_pred_upgraded.csv", index=False)
print("Saved train_df_pred_upgraded.csv")


Epoch 1/10 — loss: 0.6060
Epoch 2/10 — loss: 0.6479
Epoch 3/10 — loss: 0.6178
Epoch 4/10 — loss: 0.5967
Epoch 5/10 — loss: 0.5838
Epoch 6/10 — loss: 0.5804
Epoch 7/10 — loss: 0.5568
Epoch 8/10 — loss: 0.5511
Epoch 9/10 — loss: 0.5488
Epoch 10/10 — loss: 0.5290
Saved: submissions/submission_dualhead_upgraded_10ep.csv
Saved train_df_pred_upgraded.csv


In [ ]:
EPOCHS = 10
train_upgraded(model, train_loader, epochs=EPOCHS)

# ============================================================
# RUN PREDICTION
# ============================================================

preds_test = predict_upgraded(model, test_loader)
preds_train = predict_upgraded(model, train_loader)

os.makedirs("submissions", exist_ok=True)

sub = pd.read_csv("sample_submission.csv")
sub["score"] = preds_test
out_path = f"submissions/submission_dualhead_upgraded_{EPOCHS}ep.csv"
sub.to_csv(out_path, index=False)
print("Saved:", out_path)

train_df["pred_score"] = preds_train
train_df.to_csv("train_df_pred_upgraded.csv", index=False)
print("Saved train_df_pred_upgraded.csv")

Epoch 1/10 — loss: 0.4854
Epoch 2/10 — loss: 0.5306
Epoch 3/10 — loss: 0.5276
Epoch 4/10 — loss: 0.5237
Epoch 5/10 — loss: 0.5089
Epoch 6/10 — loss: 0.5133
Epoch 7/10 — loss: 0.4923
Epoch 8/10 — loss: 0.5184
Epoch 9/10 — loss: 0.4894
Epoch 10/10 — loss: 0.5139
Saved: submissions/submission_dualhead_upgraded_10ep.csv
Saved train_df_pred_upgraded.csv


In [ ]:
# ================================================
# ✨ CLEAN DUAL-HEAD ALIGNMENT PIPELINE (IMPROVED)
# ================================================
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH = 32
LOW_THRESHOLD = 0.5


In [ ]:
class MetricDataset(Dataset):
    def __init__(self, df, metric_embs, embedder, has_score=True):
        self.df = df.reset_index(drop=True)
        self.metric_embs = metric_embs
        self.embedder = embedder
        self.has_score = has_score

        print("Encoding dataset…")
        all_embs = []

        for _, row in tqdm(self.df.iterrows(), total=len(self.df)):
            parts = [
                str(row.get("system_prompt", "")),
                str(row.get("user_prompt", "")),
                str(row.get("response", ""))
            ]
            vecs = embedder.encode(parts, convert_to_numpy=True)
            all_embs.append(np.concatenate(vecs, axis=0))

        arr = np.vstack(all_embs)
        self.text_embs = torch.tensor(arr, dtype=torch.float32)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = self.text_embs[idx]
        metric = torch.tensor(self.metric_embs[row["metric_name"]], dtype=torch.float32)
        score = float(row["score"]) / 10.0 if self.has_score else 0.0
        return text, metric, torch.tensor(score, dtype=torch.float32)


In [ ]:
class DualHeadAlignment(nn.Module):
    def __init__(self, text_dim, metric_dim, hidden=512):
        super().__init__()

        self.text_proj = nn.Sequential(
            nn.Linear(text_dim, hidden),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.LayerNorm(hidden),
            nn.Linear(hidden, hidden)
        )

        self.metric_proj = nn.Sequential(
            nn.Linear(metric_dim, hidden),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.LayerNorm(hidden),
            nn.Linear(hidden, hidden)
        )

        self.reg_low = nn.Sequential(
            nn.Linear(text_dim + metric_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1)
        )

        self.reg_high = nn.Sequential(
            nn.Linear(text_dim + metric_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1)
        )

        self.temperature = nn.Parameter(torch.tensor(0.07))

    def embed(self, t, m):
        tp = F.normalize(self.text_proj(t), dim=-1)
        mp = F.normalize(self.metric_proj(m), dim=-1)
        return tp, mp

    def forward_low(self, t, m):
        return self.reg_low(torch.cat([t, m], dim=-1)).squeeze(-1)

    def forward_high(self, t, m):
        return self.reg_high(torch.cat([t, m], dim=-1)).squeeze(-1)


In [ ]:
def info_nce(tp, mp, temperature):
    # tp/mp: (B, D)
    logits = (tp @ mp.T) / temperature.clamp(min=1e-6)

    labels = torch.arange(len(tp), device=tp.device)
    return F.cross_entropy(logits, labels)


In [ ]:
def train_stage2(model, loader, epochs=5, lr=1e-4, low_thresh=LOW_THRESHOLD):
    freeze_encoder(model)
    model.train()

    reg_params = list(model.reg_low.parameters()) + list(model.reg_high.parameters())
    opt = torch.optim.AdamW(reg_params, lr=lr)

    for ep in range(epochs):
        total = 0

        for text, metric, score in loader:
            text, metric, score = text.to(DEVICE), metric.to(DEVICE), score.to(DEVICE)

            with torch.no_grad():
                tp, mp = model.embed(text, metric)
                sim = (tp * mp).sum(-1)

            mask_low  = score <= low_thresh
            mask_high = score > low_thresh

            loss = 0.0

            if mask_low.any():
                pred_low = torch.sigmoid(model.forward_low(text[mask_low], metric[mask_low]))
                loss += F.smooth_l1_loss(pred_low, score[mask_low])

            if mask_high.any():
                pred_high = torch.sigmoid(model.forward_high(text[mask_high], metric[mask_high]))
                loss += F.smooth_l1_loss(pred_high, score[mask_high])

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(reg_params, 1.0)
            opt.step()

            total += loss.item()

        print(f"[Stage-2] Epoch {ep+1}/{epochs} — Loss: {total/len(loader):.4f}")


In [ ]:
def freeze_encoder(model):
    for p in model.text_proj.parameters():
        p.requires_grad = False
    for p in model.metric_proj.parameters():
        p.requires_grad = False


In [ ]:
def train_stage2(model, loader, epochs=5, lr=1e-4, low_thresh=LOW_THRESHOLD):
    freeze_encoder(model)
    model.train()

    reg_params = list(model.reg_low.parameters()) + list(model.reg_high.parameters())
    opt = torch.optim.AdamW(reg_params, lr=lr)

    for ep in range(epochs):
        total = 0

        for text, metric, score in loader:
            text, metric, score = text.to(DEVICE), metric.to(DEVICE), score.to(DEVICE)

            with torch.no_grad():
                tp, mp = model.embed(text, metric)
                sim = (tp * mp).sum(-1)

            mask_low = score <= low_thresh
            mask_high = score > low_thresh

            loss = 0.0

            if mask_low.any():
                pred = torch.sigmoid(model.forward_low(text[mask_low], metric[mask_low]))
                loss += F.smooth_l1_loss(pred, score[mask_low])

            if mask_high.any():
                pred = torch.sigmoid(model.forward_high(text[mask_high], metric[mask_high]))
                loss += F.smooth_l1_loss(pred, score[mask_high])

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(reg_params, 1.0)
            opt.step()

            total += loss.item()

        print(f"[Stage-2] Epoch {ep+1}/{epochs} — Loss: {total/len(loader):.4f}")


In [ ]:
def predict(model, loader, low_thresh=LOW_THRESHOLD):
    model.eval()
    outputs = []

    with torch.no_grad():
        for text, metric, _ in loader:
            text, metric = text.to(DEVICE), metric.to(DEVICE)

            tp, mp = model.embed(text, metric)
            sim = torch.sigmoid((tp * mp).sum(-1))   # shape (B)

            mask_low  = sim <= low_thresh
            mask_high = sim > low_thresh

            if mask_low.any():
                raw_low = model.forward_low(text[mask_low], metric[mask_low])
                sim[mask_low] = torch.sigmoid(raw_low)

            if mask_high.any():
                raw_high = model.forward_high(text[mask_high], metric[mask_high])
                sim[mask_high] = torch.sigmoid(raw_high)

            outputs.extend((sim.cpu().numpy() * 10).tolist())

    return outputs


In [ ]:




model = DualHeadAlignment(text_dim, metric_dim).to(DEVICE)

# -----------------------
# Stage 1: Contrastive
# -----------------------
train_stage1(model, train_loader, epochs=6)

# -----------------------
# Stage 2: Regression
# -----------------------
train_stage2(model, train_loader, epochs=10)

# -----------------------
# Predict
# -----------------------
pred_test = predict(model, test_loader)
pred_train = predict(model, train_loader)


[Stage-1] Epoch 1/6 — Loss: 1.2886
[Stage-1] Epoch 2/6 — Loss: 0.7050
[Stage-1] Epoch 3/6 — Loss: 0.5561
[Stage-1] Epoch 4/6 — Loss: 0.4929
[Stage-1] Epoch 5/6 — Loss: 0.4501
[Stage-1] Epoch 6/6 — Loss: 0.4113
[Stage-2] Epoch 1/10 — Loss: 0.0057
[Stage-2] Epoch 2/10 — Loss: 0.0031
[Stage-2] Epoch 3/10 — Loss: 0.0029
[Stage-2] Epoch 4/10 — Loss: 0.0027
[Stage-2] Epoch 5/10 — Loss: 0.0026
[Stage-2] Epoch 6/10 — Loss: 0.0024
[Stage-2] Epoch 7/10 — Loss: 0.0023
[Stage-2] Epoch 8/10 — Loss: 0.0021
[Stage-2] Epoch 9/10 — Loss: 0.0023
[Stage-2] Epoch 10/10 — Loss: 0.0020


In [ ]:
os.makedirs("submissions", exist_ok=True)

sub = pd.read_csv("sample_submission.csv")
sub["score"] = pred_test
sub.to_csv("submissions/submission_improved.csv", index=False)
print("Saved submission_improved.csv")

train_df["pred_score"] = pred_train
train_df.to_csv("train_pred_improved.csv", index=False)
print("Saved train_pred_improved.csv")


Saved submission_improved.csv
Saved train_pred_improved.csv


In [ ]:
# ================================================
# ✨ CLEAN DUAL-HEAD ALIGNMENT (LOGIC RESTORED)
# ================================================
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

# -----------------------------
# CONFIG
# -----------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH = 32
LOW_THRESHOLD = 0.5
LR = 2e-4
EPOCHS = 10  # Single stage, more epochs

# -----------------------------
# DATASET
# -----------------------------
class MetricDataset(Dataset):
    def __init__(self, df, metric_embs, embedder, has_score=True):
        self.df = df.reset_index(drop=True)
        self.metric_embs = metric_embs
        self.embedder = embedder
        self.has_score = has_score

        print("Encoding dataset…")
        all_embs = []
        # Batch encoding is faster than iterrows
        # Construct list of texts first
        prompts = []
        for _, row in self.df.iterrows():
            parts = [
                str(row.get("system_prompt", "")),
                str(row.get("user_prompt", "")),
                str(row.get("response", ""))
            ]
            prompts.append(parts)

        # Flatten slightly for encoding to optimize, or encode loop (kept your loop for safety)
        for parts in tqdm(prompts, desc="Embedding"):
            vecs = embedder.encode(parts, convert_to_numpy=True)
            all_embs.append(np.concatenate(vecs, axis=0))

        arr = np.vstack(all_embs)
        self.text_embs = torch.tensor(arr, dtype=torch.float32)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = self.text_embs[idx]
        metric = torch.tensor(self.metric_embs[row["metric_name"]], dtype=torch.float32)
        score = float(row["score"]) / 10.0 if self.has_score else 0.0
        return text, metric, torch.tensor(score, dtype=torch.float32)

# -----------------------------
# MODEL (Restored to Simple MLP)
# -----------------------------
class DualHeadAlignment(nn.Module):
    def __init__(self, text_dim, metric_dim, hidden=512):
        super().__init__()

        # Reverted to the simpler architecture that worked
        self.text_proj = nn.Sequential(
            nn.Linear(text_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden)
        )

        self.metric_proj = nn.Sequential(
            nn.Linear(metric_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden)
        )

        # Regression Heads
        self.reg_low = nn.Sequential(
            nn.Linear(text_dim + metric_dim, hidden // 2),
            nn.ReLU(),
            nn.Linear(hidden // 2, 1)
        )

        self.reg_high = nn.Sequential(
            nn.Linear(text_dim + metric_dim, hidden // 2),
            nn.ReLU(),
            nn.Linear(hidden // 2, 1)
        )

    def embed(self, t, m):
        # No normalization here allows magnitude to play a role in regression
        # If you use Cosine Similarity, add F.normalize here.
        tp = self.text_proj(t)
        mp = self.metric_proj(m)
        return tp, mp

    def forward_reg_low(self, t, m):
        return self.reg_low(torch.cat([t, m], dim=-1)).squeeze(-1)

    def forward_reg_high(self, t, m):
        return self.reg_high(torch.cat([t, m], dim=-1)).squeeze(-1)

# -----------------------------
# JOINT TRAINING LOOP (The "Magic" Sauce)
# -----------------------------
def train_joint(model, loader, epochs=EPOCHS, lr=LR, low_thresh=LOW_THRESHOLD):
    model.train()
    # Optimize EVERYTHING at once
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)

    # Loss functions
    criterion_bce = nn.BCEWithLogitsLoss()
    criterion_mse = nn.MSELoss()

    for ep in range(epochs):
        total_loss = 0

        for text, metric, score in loader:
            text, metric, score = text.to(DEVICE), metric.to(DEVICE), score.to(DEVICE)

            optimizer.zero_grad()

            # 1. Embed
            tp, mp = model.embed(text, metric)

            # 2. Similarity (Dot Product)
            sim = (tp * mp).sum(dim=-1)

            # 3. Masks
            mask_high = score > low_thresh
            mask_low = score <= low_thresh

            # --- A. Contrastive / Alignment Losses ---

            # Positive pairs (High scores should match)
            loss_pos = 0.0
            if mask_high.any():
                loss_pos = criterion_bce(sim[mask_high], torch.ones_like(sim[mask_high]))

            # Negative pairs 1: Low scores should NOT match
            loss_neg_low = 0.0
            if mask_low.any():
                loss_neg_low = criterion_bce(sim[mask_low], torch.zeros_like(sim[mask_low]))

            # Negative pairs 2: SWAPPED METRICS (The "Hard Negative" trick)
            # We shuffle the metrics in the batch. The text should NOT match the wrong metric.
            perm = torch.randperm(metric.size(0))
            shuffled_metric = metric[perm]
            # Recalculate MP for shuffled just to be safe (or reuse projection if frozen, but we aren't freezing)
            mp_shuffled = model.metric_proj(shuffled_metric)
            sim_swap = (tp * mp_shuffled).sum(dim=-1)
            loss_neg_swap = criterion_bce(sim_swap, torch.zeros_like(sim_swap))

            # --- B. Regression Losses ---

            loss_reg = 0.0
            if mask_low.any():
                pred_low = torch.sigmoid(model.forward_reg_low(text[mask_low], metric[mask_low]))
                loss_reg += criterion_mse(pred_low, score[mask_low])

            if mask_high.any():
                pred_high = torch.sigmoid(model.forward_reg_high(text[mask_high], metric[mask_high]))
                loss_reg += criterion_mse(pred_high, score[mask_high])

            # --- C. Combine & Step ---
            # You can weight these if needed, e.g., 1.0 * align + 5.0 * reg
            loss = loss_pos + loss_neg_low + loss_neg_swap + loss_reg

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {ep+1}/{epochs} — Loss: {total_loss/len(loader):.4f}")

# -----------------------------
# PREDICTION
# -----------------------------
def predict(model, loader, low_thresh=LOW_THRESHOLD):
    model.eval()
    outputs = []

    with torch.no_grad():
        for text, metric, _ in loader:
            text, metric = text.to(DEVICE), metric.to(DEVICE)

            # Get Similarity for Routing
            tp, mp = model.embed(text, metric)
            sim = torch.sigmoid((tp * mp).sum(dim=-1))

            mask_low = sim <= low_thresh
            mask_high = sim > low_thresh

            # Copy sim to use as placeholder for predictions
            final_preds = sim.clone()

            if mask_low.any():
                raw_low = model.forward_reg_low(text[mask_low], metric[mask_low])
                final_preds[mask_low] = torch.sigmoid(raw_low)

            if mask_high.any():
                raw_high = model.forward_reg_high(text[mask_high], metric[mask_high])
                final_preds[mask_high] = torch.sigmoid(raw_high)

            outputs.extend((final_preds.cpu().numpy() * 10).tolist())

    return outputs

# ================================================
# EXECUTION
# ================================================
# Assuming text_dim, metric_dim, train_loader, test_loader exist from your snippet

model = DualHeadAlignment(text_dim, metric_dim).to(DEVICE)
train_joint(model, train_loader, epochs=30)
preds = predict(model, test_loader)

Epoch 1/30 — Loss: 1.2407
Epoch 2/30 — Loss: 0.9420
Epoch 3/30 — Loss: 0.8265
Epoch 4/30 — Loss: 0.7654
Epoch 5/30 — Loss: 0.7090
Epoch 6/30 — Loss: 0.6538
Epoch 7/30 — Loss: 0.6500
Epoch 8/30 — Loss: 0.6501
Epoch 9/30 — Loss: 0.6212
Epoch 10/30 — Loss: 0.6217
Epoch 11/30 — Loss: 0.5799
Epoch 12/30 — Loss: 0.5675
Epoch 13/30 — Loss: 0.5702
Epoch 14/30 — Loss: 0.5538
Epoch 15/30 — Loss: 0.5606
Epoch 16/30 — Loss: 0.5436
Epoch 17/30 — Loss: 0.5587
Epoch 18/30 — Loss: 0.5013
Epoch 19/30 — Loss: 0.5141
Epoch 20/30 — Loss: 0.5259
Epoch 21/30 — Loss: 0.5220
Epoch 22/30 — Loss: 0.5257
Epoch 23/30 — Loss: 0.4728
Epoch 24/30 — Loss: 0.4776
Epoch 25/30 — Loss: 0.4914
Epoch 26/30 — Loss: 0.4903
Epoch 27/30 — Loss: 0.4858
Epoch 28/30 — Loss: 0.4960
Epoch 29/30 — Loss: 0.4697
Epoch 30/30 — Loss: 0.5006


In [ ]:
# ================================================
# SAVE PREDICTIONS
# ================================================
import os

# Make dir if needed
os.makedirs("submissions", exist_ok=True)

# ---------- SAVE TEST PREDICTIONS ----------
sub = pd.read_csv("sample_submission.csv")
sub["score"] = preds  # preds from predict(model, test_loader)
sub_path = "submissions/submission_improved.csv"
sub.to_csv(sub_path, index=False)
print(f"Saved {sub_path}")

# ---------- GENERATE TRAIN PREDICTIONS ----------
# IMPORTANT: run prediction on train_loader too
preds_train = predict(model, train_loader)

train_df["pred_score"] = preds_train
train_train_path = "train_pred_improved.csv"
train_df.to_csv(train_train_path, index=False)
print(f"Saved {train_train_path}")


Saved submissions/submission_improved.csv
Saved train_pred_improved.csv


In [ ]:
from google.colab import drive
drive.mount('/content/drive')